In [0]:

# ============================================================
# CELL 1 — Read Bronze table + fetch live exchange rates
# ============================================================
# We read all transactions from the Bronze table and fetch
# live exchange rates to normalize all amounts to USD.
# This lets us fairly compare a 50,000 JPY transaction
# with a 500 USD transaction on the same scale.

import requests
from pyspark.sql.functions import col, udf, when, lit, current_timestamp
from pyspark.sql.types import DoubleType, StringType

# -- Step 1: Read Bronze table -------------------------------
bronze_df = spark.table("aml_pipeline.transactions.bronze_transactions")
print(f"✅ Loaded {bronze_df.count()} records from Bronze table")

# -- Step 2: Fetch live exchange rates -----------------------
# We use the European Central Bank's free API — no key needed!
# It returns today's exchange rates for all major currencies vs EUR.
print("\n📡 Fetching live exchange rates from ECB API...")

try:
    ecb_url = "https://api.frankfurter.app/latest?from=USD"
    response = requests.get(ecb_url, timeout=10)
    rates_data = response.json()
    
    # Build a rates dictionary — how many units of X = 1 USD
    exchange_rates = rates_data.get("rates", {})
    exchange_rates["USD"] = 1.0  # USD to USD is always 1
    
    print(f"✅ Got exchange rates for {len(exchange_rates)} currencies")
    print(f"   Sample rates from 1 USD:")
    for currency in ["EUR", "GBP", "JPY", "CHF", "CAD"]:
        rate = exchange_rates.get(currency, "N/A")
        print(f"   1 USD = {rate} {currency}")

except Exception as e:
    print(f"⚠️  Could not fetch live rates, using fallback rates: {e}")
    exchange_rates = {
        "USD": 1.0, "EUR": 0.92, "GBP": 0.79,
        "JPY": 149.5, "CHF": 0.89, "CAD": 1.36,
        "AUD": 1.53, "SGD": 1.34
    }

# -- Step 3: Create a UDF to convert any amount to USD -------
# UDF = User Defined Function — a custom Python function
# that runs on every row in your Spark DataFrame
def convert_to_usd(amount, currency):
    """Convert any amount to USD using today's exchange rate."""
    if amount is None or currency is None:
        return None
    rate = exchange_rates.get(currency, 1.0)
    # rate = how many foreign units = 1 USD
    # so amount_in_usd = amount / rate
    return round(float(amount) / float(rate), 2)

convert_to_usd_udf = udf(convert_to_usd, DoubleType())

# -- Step 4: Add USD amount column to our dataframe ----------
bronze_with_usd = bronze_df.withColumn(
    "amount_usd",
    convert_to_usd_udf(col("amount"), col("currency"))
)

print(f"\n✅ Exchange rate conversion ready!")
print(f"Sample — showing amount vs amount_usd:")
bronze_with_usd.select(
    "sender_name", "amount", "currency", "amount_usd"
).show(5, truncate=False)

✅ Loaded 100 records from Bronze table

📡 Fetching live exchange rates from ECB API...
✅ Got exchange rates for 30 currencies
   Sample rates from 1 USD:
   1 USD = 0.85027 EUR
   1 USD = 0.73472 GBP
   1 USD = 156.76 JPY
   1 USD = 0.77851 CHF
   1 USD = 1.3658 CAD

✅ Exchange rate conversion ready!
Sample — showing amount vs amount_usd:
+-------------+--------+--------+----------+
|sender_name  |amount  |currency|amount_usd|
+-------------+--------+--------+----------+
|Maria Santos |34297.53|JPY     |218.79    |
|Alice Johnson|23784.45|GBP     |32372.13  |
|Priya Patel  |39153.52|EUR     |46048.34  |
|Alice Johnson|25989.69|EUR     |30566.4   |
|James Smith  |14070.35|GBP     |19150.63  |
+-------------+--------+--------+----------+
only showing top 5 rows


In [0]:

# ============================================================
# CELL 2 — OFAC Sanctions Screening + FATF Travel Rule Check
# ============================================================
# For every transaction we check:
#   1. Is the sender or receiver on the OFAC sanctions list?
#   2. Are all FATF Travel Rule required fields present?
#   3. Is the amount suspiciously large (>$50,000 USD)?
#   4. Is the sender from a high-risk country?

import requests
import re
from pyspark.sql.functions import udf, col, when, lit
from pyspark.sql.types import StringType, BooleanType

# -- Step 1: Download OFAC SDN sanctions list ----------------
# The US Treasury publishes this list publicly and free.
# It contains 6,000+ sanctioned individuals and entities.
print("📡 Downloading OFAC sanctions list from US Treasury...")

OFAC_URL = "https://www.treasury.gov/ofac/downloads/sdn.csv"
SANCTIONED_NAMES = set()

try:
    response = requests.get(OFAC_URL, timeout=15)
    lines = response.text.split("\n")
    for line in lines[:500]:  # First 500 entries for speed
        parts = line.split(",")
        if len(parts) > 1:
            name = parts[1].strip().strip('"').lower()
            if name:
                SANCTIONED_NAMES.add(name)
    print(f"✅ Loaded {len(SANCTIONED_NAMES)} sanctioned names from OFAC")
except Exception as e:
    print(f"⚠️  Using sample OFAC list (API unavailable): {e}")
    # Fallback — known sanctioned names from our generator
    SANCTIONED_NAMES = {
        "viktor bout", "semion mogilevich", "joaquin guzman loera",
        "alisher usmanov", "ramzan kadyrov", "ali khamenei",
        "kim jong un", "robert mugabe"
    }
    print(f"✅ Loaded {len(SANCTIONED_NAMES)} sanctioned names (fallback)")

# -- Step 2: Define high-risk countries (FATF list) ----------
HIGH_RISK_COUNTRIES = {"KP", "IR", "MM", "RU", "BY", "CU", "SY", "YE"}

# -- Step 3: Create screening UDFs ---------------------------

def check_sanctions(name):
    """Check if a name matches the OFAC sanctions list."""
    if not name:
        return "CLEAN"
    name_lower = name.lower().strip()
    # Direct match
    if name_lower in SANCTIONED_NAMES:
        return "SANCTIONS_HIT"
    # Partial match — catches variations of sanctioned names
    for sanctioned in SANCTIONED_NAMES:
        if sanctioned and len(sanctioned) > 5:
            if sanctioned in name_lower or name_lower in sanctioned:
                return "SANCTIONS_HIT"
    return "CLEAN"

def check_travel_rule(sender_address, sender_name, sender_account,
                      receiver_name, receiver_account):
    """
    FATF Travel Rule: all identity fields must be present.
    Missing any field = compliance violation.
    """
    missing = []
    if not sender_address or sender_address.strip() == "":
        missing.append("sender_address")
    if not sender_name or sender_name.strip() == "":
        missing.append("sender_name")
    if not sender_account or sender_account.strip() == "":
        missing.append("sender_account")
    if not receiver_name or receiver_name.strip() == "":
        missing.append("receiver_name")
    if not receiver_account or receiver_account.strip() == "":
        missing.append("receiver_account")
    if missing:
        return f"TRAVEL_RULE_VIOLATION: missing {', '.join(missing)}"
    return "COMPLIANT"

# Register as Spark UDFs
sanctions_udf     = udf(check_sanctions, StringType())
travel_rule_udf   = udf(check_travel_rule, StringType())

# -- Step 4: Apply all checks to every transaction -----------
print("\n🔍 Running compliance checks on all transactions...")

screened_df = (
    bronze_with_usd

    # OFAC check on sender
    .withColumn("sender_sanctions_status",
        sanctions_udf(col("sender_name")))

    # OFAC check on receiver
    .withColumn("receiver_sanctions_status",
        sanctions_udf(col("receiver_name")))

    # FATF Travel Rule check
    .withColumn("travel_rule_status",
        travel_rule_udf(
            col("sender_address"),
            col("sender_name"),
            col("sender_account"),
            col("receiver_name"),
            col("receiver_account")
        ))

    # High-risk country flag
    .withColumn("high_risk_country",
        col("sender_country").isin(list(HIGH_RISK_COUNTRIES)))

    # Large transaction flag (>$50,000 USD)
    .withColumn("large_transaction",
        col("amount_usd") > 50000)

    # Overall risk flag — flagged if ANY check fails
    .withColumn("is_flagged",
        (col("sender_sanctions_status") == "SANCTIONS_HIT") |
        (col("receiver_sanctions_status") == "SANCTIONS_HIT") |
        (col("travel_rule_status") != "COMPLIANT") |
        (col("high_risk_country") == True) |
        (col("large_transaction") == True)
    )

    # Risk reason — explains WHY it was flagged
    .withColumn("flag_reason",
        when(col("sender_sanctions_status") == "SANCTIONS_HIT",
             lit("OFAC: Sanctioned sender"))
        .when(col("receiver_sanctions_status") == "SANCTIONS_HIT",
             lit("OFAC: Sanctioned receiver"))
        .when(col("travel_rule_status") != "COMPLIANT",
             col("travel_rule_status"))
        .when(col("high_risk_country") == True,
             lit("HIGH_RISK_COUNTRY"))
        .when(col("large_transaction") == True,
             lit("LARGE_TRANSACTION_>50K_USD"))
        .otherwise(lit("NONE"))
    )
)

# -- Step 5: Show results ------------------------------------
total     = screened_df.count()
flagged   = screened_df.filter(col("is_flagged") == True).count()
clean     = total - flagged

print(f"\n📊 Screening Results:")
print(f"   Total transactions : {total}")
print(f"   ✅ Clean           : {clean}")
print(f"   🚨 Flagged         : {flagged}")
print(f"\n🚨 Flagged transactions:")
screened_df.filter(col("is_flagged") == True) \
    .select("sender_name", "receiver_name", "amount_usd",
            "sender_country", "is_flagged", "flag_reason") \
    .show(20, truncate=False)

📡 Downloading OFAC sanctions list from US Treasury...
✅ Loaded 470 sanctioned names from OFAC

🔍 Running compliance checks on all transactions...

📊 Screening Results:
   Total transactions : 100
   ✅ Clean           : 80
   🚨 Flagged         : 20

🚨 Flagged transactions:
+-------------+-------------+----------+--------------+----------+---------------------------------------------+
|sender_name  |receiver_name|amount_usd|sender_country|is_flagged|flag_reason                                  |
+-------------+-------------+----------+--------------+----------+---------------------------------------------+
|Liam Brown   |Alice Johnson|62580.64  |DE            |true      |LARGE_TRANSACTION_>50K_USD                   |
|Alice Johnson|Wei Zhang    |51194.77  |CA            |true      |LARGE_TRANSACTION_>50K_USD                   |
|Liam Brown   |Emma Wilson  |3433126.87|IR            |true      |HIGH_RISK_COUNTRY                            |
|James Smith  |Wei Zhang    |51358.07  |DE       

In [0]:

# ============================================================
# CELL 3 — Write enriched data to Silver Delta Table
# ============================================================
# The Silver table is the "cleaned and enriched" layer of our
# Medallion Architecture. Every transaction now has:
#   - Original fields from Bronze
#   - amount_usd (normalized to USD using live rates)
#   - sanctions screening results (OFAC)
#   - Travel Rule compliance status (FATF)
#   - High-risk country flag
#   - Large transaction flag
#   - is_flagged (overall risk flag)
#   - flag_reason (explains exactly why it was flagged)
#
# Think of Bronze as the raw security camera footage,
# and Silver as the footage with faces identified,
# annotations added, and suspicious moments highlighted.

from pyspark.sql.functions import current_timestamp, lit

SILVER_TABLE = "aml_pipeline.transactions.silver_transactions"

# -- Add Silver layer metadata columns ----------------------
silver_df = (
    screened_df
    .withColumn("silver_timestamp", current_timestamp())
    .withColumn("pipeline_layer",   lit("silver"))
)

# -- Write to Silver Delta Table ----------------------------
# We use overwrite mode here since we're processing a full
# batch. In production this would be a merge/upsert.
print("💾 Writing to Silver Delta table...")

(
    silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(SILVER_TABLE)
)

# -- Verify --------------------------------------------------
count        = spark.table(SILVER_TABLE).count()
flagged      = spark.table(SILVER_TABLE).filter("is_flagged = true").count()
clean        = count - flagged

print(f"✅ Silver table written successfully!")
print(f"\n📊 Silver Table Summary:")
print(f"   Table        : {SILVER_TABLE}")
print(f"   Total records: {count}")
print(f"   ✅ Clean     : {clean}")
print(f"   🚨 Flagged   : {flagged}")
print(f"   Flag rate    : {round(flagged/count*100, 1)}%")

print(f"\n📋 Flag reason breakdown:")
spark.table(SILVER_TABLE) \
     .groupBy("flag_reason") \
     .count() \
     .orderBy("count", ascending=False) \
     .show(truncate=False)

print(f"\n🏷️  Silver layer columns:")
for col_name in spark.table(SILVER_TABLE).columns:
    print(f"   • {col_name}")

💾 Writing to Silver Delta table...
✅ Silver table written successfully!

📊 Silver Table Summary:
   Table        : aml_pipeline.transactions.silver_transactions
   Total records: 100
   ✅ Clean     : 80
   🚨 Flagged   : 20
   Flag rate    : 20.0%

📋 Flag reason breakdown:
+---------------------------------------------+-----+
|flag_reason                                  |count|
+---------------------------------------------+-----+
|NONE                                         |80   |
|LARGE_TRANSACTION_>50K_USD                   |17   |
|HIGH_RISK_COUNTRY                            |2    |
|TRAVEL_RULE_VIOLATION: missing sender_address|1    |
+---------------------------------------------+-----+


🏷️  Silver layer columns:
   • transaction_id
   • timestamp
   • message_type
   • sender_name
   • sender_account
   • sender_country
   • sender_address
   • receiver_name
   • receiver_account
   • receiver_country
   • amount
   • currency
   • purpose_code
   • transaction_type
   • ing